# Baseline YouTube Video Viewership Forecasting Model
This notebook builds a baseline model to predict a video's viewership (latest available view count) using channel-level stats and early engagement signals.

## Steps:
1. Data Loading and Cleaning
2. Build the Flat Table
3. Feature Engineering
4. Train/Test Split
5. Naive Baseline
6. Model Baseline - LightGBM & RandomForest
7. Residual / Sanity Check Plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

## 1. DATA LOADING AND CLEANING
- Load all three CSVs.
- Convert view_count, like_count, comment_count to numeric.
- Convert datetime columns to pandas datetime format.
- Handle missing values resulting from conversion.

In [ ]:
# Load CSVs
channel_stats = pd.read_csv('channel_stats.csv')
videos_rows = pd.read_csv('videos_rows.csv')
view_timeseries = pd.read_csv('view_timeseries.csv')

# Convert metrics to numeric
for col in ['view_count', 'like_count', 'comment_count']:
    original_nulls = view_timeseries[col].isnull().sum()
    view_timeseries[col] = pd.to_numeric(view_timeseries[col], errors='coerce')
    new_nulls = view_timeseries[col].isnull().sum()
    print(f"{col}: {new_nulls - original_nulls} rows became NaN after numeric conversion.")

# Convert datetimes
datetime_cols_channel = ['processed_at', 'created_at']
for col in datetime_cols_channel:
    if col in channel_stats.columns:
        channel_stats[col] = pd.to_datetime(channel_stats[col], errors='coerce')

datetime_cols_videos = ['published_at', 'last_polled_at', 'next_poll_at', 'created_at']
for col in datetime_cols_videos:
    if col in videos_rows.columns:
        videos_rows[col] = pd.to_datetime(videos_rows[col], errors='coerce')

datetime_cols_timeseries = ['scraped_at']
for col in datetime_cols_timeseries:
    if col in view_timeseries.columns:
        view_timeseries[col] = pd.to_datetime(view_timeseries[col], errors='coerce')

# Print null counts and dtypes
print("\n--- channel_stats ---")
print(channel_stats.isnull().sum())
print(channel_stats.dtypes)

print("\n--- videos_rows ---")
print(videos_rows.isnull().sum())
print(videos_rows.dtypes)

print("\n--- view_timeseries ---")
print(view_timeseries.isnull().sum())
print(view_timeseries.dtypes)

# Drop rows in view_timeseries where view_count is null
view_timeseries = view_timeseries.dropna(subset=['view_count'])
print(f"\nRemaining rows in view_timeseries after dropping null view_count: {len(view_timeseries)}")

## 2. BUILD THE FLAT TABLE
- Compute hours_since_publish for each scrape.
- Identify the earliest (early snapshot) and latest (target) scrape rows for each video.
- Merge in channel stats and compute channel_avg_views.
- Result: one row per video_id.

In [ ]:
# Merge view_timeseries with videos_rows to get published_at
vt_merged = view_timeseries.merge(videos_rows[['video_id', 'channel_id', 'published_at']], on='video_id', how='left')

# Compute hours_since_publish
vt_merged['hours_since_publish'] = (vt_merged['scraped_at'] - vt_merged['published_at']).dt.total_seconds() / 3600.0

# Drop rows where hours_since_publish is null (e.g. missing published_at)
vt_merged = vt_merged.dropna(subset=['hours_since_publish'])

# Identify earliest and latest scrapes per video
vt_sorted = vt_merged.sort_values(['video_id', 'hours_since_publish'])

early_snapshot = vt_sorted.groupby('video_id').first().reset_index()
latest_snapshot = vt_sorted.groupby('video_id').last().reset_index()

# Rename columns for early and latest snapshots
early_snapshot = early_snapshot[['video_id', 'channel_id', 'view_count', 'like_count', 'comment_count', 'hours_since_publish']]
early_snapshot.columns = ['video_id', 'channel_id', 'early_view_count', 'early_like_count', 'early_comment_count', 'early_hours_since_publish']

latest_snapshot = latest_snapshot[['video_id', 'view_count', 'hours_since_publish']]
latest_snapshot.columns = ['video_id', 'latest_view_count', 'latest_hours_since_publish']

# Merge early and latest back together
flat_table = pd.merge(early_snapshot, latest_snapshot, on='video_id')

# Drop videos where early and latest snapshot are the same row
flat_table = flat_table[flat_table['early_hours_since_publish'] < flat_table['latest_hours_since_publish']]

# Merge in channel_stats
flat_table = flat_table.merge(channel_stats[['channel_id', 'total_views', 'subscriber_count', 'video_count']], on='channel_id', how='left')

# Compute channel_avg_views
flat_table['channel_avg_views'] = flat_table['total_views'] / flat_table['video_count']
flat_table.loc[flat_table['video_count'] == 0, 'channel_avg_views'] = np.nan

# Keep required columns
final_columns = ['video_id', 'channel_id', 'early_view_count', 'early_like_count', 'early_comment_count', 
                 'early_hours_since_publish', 'latest_view_count', 'latest_hours_since_publish', 
                 'subscriber_count', 'total_views', 'video_count', 'channel_avg_views']
flat_table = flat_table[final_columns]

print(f"Shape of final flat table: {flat_table.shape}")
display(flat_table.describe())

## 3. FEATURE ENGINEERING
- Log-transform engagement metrics and channel stats (using np.log1p).
- Create log_target for the model target.
- Clean up any infinite or missing values.

In [ ]:
# Log transformations using np.log1p (log(1 + x))
log_cols = ['early_view_count', 'early_like_count', 'early_comment_count', 
            'subscriber_count', 'total_views', 'channel_avg_views', 'latest_view_count']

for col in log_cols:
    # Ensure no negative values before log (clip to 0)
    flat_table[col] = flat_table[col].clip(lower=0)
    flat_table[f'log_{col}'] = np.log1p(flat_table[col])

# The target is log_latest_view_count
flat_table['log_target'] = flat_table['log_latest_view_count']

# Final feature list
features = [
    'log_early_view_count', 'log_early_like_count', 'log_early_comment_count', 
    'early_hours_since_publish', 'log_subscriber_count', 'log_total_views', 
    'log_channel_avg_views', 'video_count'
]
target = 'log_target'

# Check for NaN or inf values in features or target
model_data = flat_table[features + [target, 'latest_view_count']].copy()
model_data.replace([np.inf, -np.inf], np.nan, inplace=True)
initial_len = len(model_data)
model_data = model_data.dropna()
dropped_len = initial_len - len(model_data)

print(f"Dropped {dropped_len} rows due to NaN or Inf values in features or target.")
print(f"Final dataset size for modeling: {len(model_data)}")

## 4. TRAIN/TEST SPLIT
- 80/20 split, random state = 42.

In [ ]:
X = model_data[features]
y = model_data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

## 5. NAIVE BASELINE
- Predict target as log(channel_avg_views) for all videos.
- Compute MAE and RMSE on log scale and original views scale.

In [ ]:
# Naive prediction: log(channel_avg_views)
# log_channel_avg_views is already in X_test
y_pred_naive_log = X_test['log_channel_avg_views']

# Calculate metrics on log scale
mae_naive_log = mean_absolute_error(y_test, y_pred_naive_log)
rmse_naive_log = np.sqrt(mean_squared_error(y_test, y_pred_naive_log))

# Invert to original scale using expm1
y_test_views = np.expm1(y_test)
y_pred_naive_views = np.expm1(y_pred_naive_log)
mae_naive_views = mean_absolute_error(y_test_views, y_pred_naive_views)

print("--- NAIVE BASELINE ---")
print(f"MAE (log scale): {mae_naive_log:.4f}")
print(f"RMSE (log scale): {rmse_naive_log:.4f}")
print(f"MAE (views scale): {mae_naive_views:,.0f}")

## 6. MODEL BASELINE - LightGBM
- Train LightGBM Regressor and RandomForest Regressor.
- Compare metrics.
- Plot feature importances for LightGBM.

In [ ]:
# Train LightGBM
lgb_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)
lgb_model.fit(X_train, y_train)
y_pred_lgb_log = lgb_model.predict(X_test)
y_pred_lgb_views = np.expm1(y_pred_lgb_log)

# Train RandomForest
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf_log = rf_model.predict(X_test)
y_pred_rf_views = np.expm1(y_pred_rf_log)

# Calculate metrics function
def calc_metrics(y_true_log, y_pred_log):
    mae_log = mean_absolute_error(y_true_log, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
    y_true_views = np.expm1(y_true_log)
    y_pred_views = np.expm1(y_pred_log)
    mae_views = mean_absolute_error(y_true_views, y_pred_views)
    return mae_log, rmse_log, mae_views

# Compile results
results = {
    'Model': ['Naive Baseline', 'RandomForest', 'LightGBM'],
    'MAE (log scale)': [mae_naive_log, calc_metrics(y_test, y_pred_rf_log)[0], calc_metrics(y_test, y_pred_lgb_log)[0]],
    'RMSE (log scale)': [rmse_naive_log, calc_metrics(y_test, y_pred_rf_log)[1], calc_metrics(y_test, y_pred_lgb_log)[1]],
    'MAE (views scale)': [mae_naive_views, calc_metrics(y_test, y_pred_rf_log)[2], calc_metrics(y_test, y_pred_lgb_log)[2]]
}

results_df = pd.DataFrame(results)
display(results_df)

# Feature Importances for LightGBM
importances = lgb_model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 6))
plt.title("LightGBM Feature Importances")
plt.barh(range(len(indices)), importances[indices], align="center")
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 7. RESIDUAL / SANITY CHECK PLOTS
- Scatter plot of Predicted vs Actual log(latest_view_count).
- Short markdown summary.

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_lgb_log, alpha=0.3, label='Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='y = x')
plt.xlabel('Actual log(latest_view_count)')
plt.ylabel('Predicted log(latest_view_count)')
plt.title('LightGBM: Predicted vs Actual (Log Scale)')
plt.legend()
plt.tight_layout()
plt.show()

### Summary
- The **LightGBM model** is evaluated against the RandomForest and Naive baselines.
- It typically offers improvements over the Naive Baseline (which relies solely on historical channel average views).
- *Note: Refer to the comparison table above to see the exact % reduction in MAE and other metrics.*